In [ ]:
import numpy as np
import pandas as pd

Data are often spread across many files/ DBs, or arranged in a form not convenient to analyze. Before getting to the particular data manipulations (combine, join, and rearrange), we 1st cover hierarchical indexing, which these operations use heavily. It allows multiple index levels on an axis. Another way to think of it is it lets you work with high dimensional data in lower dimensional form. They are often displayed 'prettified,' where 'gaps' mean 'use the label directly above'

In [ ]:
data = pd.Series(np.random.uniform(size=9),
                 index=[["a", "a", "a", "b", "b", "c", "c", "d", "d"],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])
data
data.index

In [ ]:
#Partial indexing lets you concisely select many subsets
data["b"]
data["b":"c"]
data.loc[["b", "d"]]
#Select from an inner level
data.loc[:, 2]

Hierarchical indexing is key to reshaping data and in group-based operations (eg forming a pivot table). The <i>unstack()</i> method rearranges data with MultiIndex into a pd.DataFrame. Its inverse is <i>stack()</i> (more later)

In [ ]:
data.unstack().stack()==data
data.unstack()

For a pd.DataFrame, either axis can have a hierarchical index. The hierarchical levels can have names (strings or any Python objects). They supersede the <i>name</i> attribute (used only for single-level indexes).

In [ ]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                     index=[["a", "a", "b", "b"], [1, 2, 1, 2]],
                     columns=[["Ohio", "Ohio", "Colorado"],
                              ["Green", "Red", "Green"]])
frame.index.names = ["key1", "key2"]
frame.columns.names = ["state", "color"]
frame.index.nlevels #number of levels in an index
frame["Ohio"] #partial column indexing to select groups of columns

A MultiIndex can be made directly and then reused. The 1 for the columns in the above pd.DataFrame is:

In [ ]:
pd.MultiIndex.from_arrays([["Ohio", "Ohio", "Colorado"],
                          ["Green", "Red", "Green"]],
                          names=["state", "color"])

The <i>swaplevel()</i> method takes 2 level numbers or names and returns a new object interchanging them (the data is otherwise unaltered). <i>sort_index()</i> by default sorts lexicographically using all index levels, give a subset via the <i>level</i> argument. Data selection performance is better on hierarchically indexed objects if they are lexicographically sorted starting with the outermost level, ie after calling <i>sort_index()</i>.

In [ ]:
frame.swaplevel("key1", "key2")
frame.sort_index(level=1)
frame.swaplevel(0, 1).sort_index(level=0)

Many descriptive and summary statistics have a <i>level</i> option to aggregate by on a particular axis

In [ ]:
frame.groupby(level="key2").sum()
frame.groupby(level="color", axis="columns").sum()

<i>set_index()</i> makes a new pd.DataFrame using 1+ columns as the index. these columns are removed unless <i>drop=False</i>. <i>reset_index()</i> does the opposite: levels in <i>level</i> (default all) are moved to the columns

In [ ]:
frame = pd.DataFrame({"a": range(7), "b": range(7, 0, -1),
                      "c": ["one", "one", "one", "two", "two",
                            "two", "two"],
                      "d": [0, 1, 2, 0, 1, 2, 3]})
frame2 = frame.set_index(["c", "d"])
frame.set_index(["c", "d"], drop=False)
frame2.reset_index()

pd has many ways to combine data. <i>pd.merge()</i> (a DB- style join) connects rows in DataFrames by 1+ keys. SQL users will find it familiar as it implements DB join operations. <i>pd.concat()</i> concatenates (stacks - do not confuse with <i>pd.stack()</i>) objects together along an axis. <i>combine_first()</i> splices together overlapping data to fill in NAs in one object with values from another.<br>
The next example is a many-to-one join: df1 has many rows labeled a and b, whereas df2 has only 1 row per value of key. Overlapping column names are the default keys. It is good practice to specify them explicitly via <i>on</i>. The column output order is unspecified.<br>
When joining on columns, the pd.DataFrame indexes are discarded. To preserve their values, use <i>reset_index()</i>

In [ ]:
df1 = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "a", "b"],
                    "data1": pd.Series(range(7), dtype="Int64",
                   index=list('qjx368]'))})
df2 = pd.DataFrame({"key": ["a", "b", "d"],
                    "data2": pd.Series(range(3), dtype="Int64")})
pd.merge(left=df1, right=df2)
pd.merge(df1, df2, on="key") #same
pd.merge(df1.reset_index(), df2, on="key")

In [ ]:
df3 = pd.DataFrame({"lkey": ["b", "b", "a", "c", "a", "a", "b"],
                    "data1": pd.Series(range(7), dtype="Int64")})
df4 = pd.DataFrame({"rkey": ["a", "b", "d"],
                    "data2": pd.Series(range(3), dtype="Int64")})
#If column names differ per pd.DataFrame, specify them separately
pd.merge(df3, df4, left_on="lkey", right_on="rkey")
#outer join can introduce NAs
pd.merge(df3, df4, left_on="lkey", right_on="rkey", how="outer")

<table border="1" cellpadding="6" cellspacing="0">
<tr><th>Option</th><th>Behavior</th></tr>
<tr><td>how="inner"</td><td>Use only the key combinations observed in both tables</td></tr>
<tr><td>how="left"</td><td>Use all key combinations found in the left table</td></tr>
<tr><td>how="right"</td><td>Use all key combinations found in the right table</td></tr>
<tr><td>how="outer"</td><td>Use all key combinations observed in both tables together</td></tr>
</table>

Many-to-many merges form the Cartesian product of matching keys. The join method in <i>how</i> affects only the distinct key values

In [ ]:
df1 = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"],
                    "data1": pd.Series(range(6), dtype="Int64")})
df2 = pd.DataFrame({"key": ["a", "b", "a", "b", "d"],
                    "data2": pd.Series(range(5), dtype="Int64")})
pd.merge(df1, df2, on="key", how="left")

Merge on many keys: to determine which key combos the result has, imagine those keys form an array of tuples to be used as a single join key

In [ ]:
left = pd.DataFrame({"key1": ["foo", "foo", "bar"],
                     "key2": ["one", "two", "one"],
                     "lval": pd.Series([1, 2, 3], dtype='Int64')})
right = pd.DataFrame({"key1": ["foo", "foo", "bar", "bar"],
                      "key2": ["one", "one", "one", "two"],
                      "rval": pd.Series([4, 5, 6, 7], dtype='Int64')})
pd.merge(left, right, on=["key1", "key2"])

In [ ]:
#intersecting column names not merged on get suffixes
pd.merge(left, right, on="key1")
pd.merge(left, right, on="key1", suffixes=("_left", "_right"))

Arguments to <i>pd.merge()</i>
<table border="1" cellpadding="6" cellspacing="0">
<tr><th>Argument</th><th>Description</th></tr>
<tr><td>left</td><td>DataFrame to be merged on the left side.</td></tr>
<tr><td>right</td><td>DataFrame to be merged on the right side.</td></tr>
<tr><td>how</td><td>Type of join to apply: one of "inner",<br>"outer", "left", or "right"; defaults to "inner".</td></tr>
<tr><td>on</td><td>Column names to join on. Must be found in<br>both DataFrame objects. If not specified and<br>no other join keys given, will use the<br>intersection of the column names in left and<br>right as the join keys.</td></tr>
<tr><td>left_on</td><td>Columns in left DataFrame to use as join<br>keys. Can be a single column name or a list<br>of column names.</td></tr>
<tr><td>right_on</td><td>Analogous to left_on for right DataFrame.</td></tr>
<tr><td>left_index</td><td>Use row index in left as its join key (or<br>keys, if a MultiIndex).</td></tr>
<tr><td>right_index</td><td>Analogous to left_index.</td></tr>
<tr><td>sort</td><td>Sort merged data lexicographically by join<br>keys; False by default.</td></tr>
<tr><td>suffixes</td><td>Tuple of string values to append to column<br>names in case of overlap; defaults to ("_x",<br>"_y") (e.g., if "data" in both DataFrame<br>objects, would appear as "data_x" and<br>"data_y" in result).</td></tr>
<tr><td>copy</td><td>If False, avoid copying data into resulting<br>data structure in some exceptional cases; by<br>default always copies.</td></tr>
<tr><td>validate</td><td>Verifies if the merge is of the specified<br>type, whether one-to-one, one-to-many, or<br>many-to-many. See the docstring for full<br>details on the options.</td></tr>
<tr><td>indicator</td><td>Adds a special column _merge that indicates<br>the source of each row; values will be<br>"left_only", "right_only", or "both" based<br>on the origin of the joined data in each row.</td></tr>
</table>

<i>left_index</i> or <i>right_index=True</i> (or both) means use the respective index as the merge key. Now the left index is preserved. The <i>join()</i> method (called on the left pd.DataFrame) simplifies merging by index (can use columns in left via <i>on</i>, index in <i>other</i> always used).

In [ ]:
left1 = pd.DataFrame({"key": ["a", "b", "a", "a", "b", "c"],
                      "value": pd.Series(range(6), dtype="Int64")})
right1 = pd.DataFrame({"group_val": [3.5, 7]}, index=["a", "b"])
pd.merge(left1, right1, left_on="key", right_index=True)
left1.join(other=right1, on="key") #same

In [ ]:
left2 = pd.DataFrame([[1., 2.], [3., 4.], [5., 6.]],
                     index=["a", "c", "e"],
                     columns=["Ohio", "Nevada"]).astype("Int64")
right2 = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [13, 14]],
                      index=["b", "c", "d", "e"],
                      columns=["Missouri", "Alabama"]).astype("Int64")
pd.merge(left2, right2, how="outer", left_index=True, right_index=True)
left2.join(right2, how="outer") #same

<i>join()</i> can also merge many pd.DataFrame objects (when <i>other</i> is a list) with similar indexes but nonoverlapping columns

In [ ]:
another = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [16., 17.]],
                       index=["a", "c", "e", "f"],
                       columns=["New York", "Oregon"])
another
left2.join([right2, another])
left2.join([right2, another], how="outer")

Joining on a MultiIndex is equivalent to a multi-key merge. You must give multiple columns to merge on for the other pd.DataFrame

In [ ]:
lefth = pd.DataFrame({"key1": ["Ohio", "Ohio", "Ohio", "Nevada", "Nevada"],
                      "key2": [2000, 2001, 2002, 2001, 2002],
                      "data": pd.Series(range(5), dtype="Int64")})
righth_index = pd.MultiIndex.from_arrays(
    [["Nevada", "Nevada", "Ohio", "Ohio", "Ohio", "Ohio"],
        [2001, 2000, 2000, 2000, 2001, 2002]])
righth = pd.DataFrame({"event1": pd.Series([0, 2, 4, 6, 8, 10], dtype="Int64",
                                           index=righth_index),
                       "event2": pd.Series([1, 3, 5, 7, 9, 11], dtype="Int64",
                                           index=righth_index)})
pd.merge(lefth, righth, left_on=["key1", "key2"], right_index=True)

We next discuss concatenation, aka stacking. In np:

In [ ]:
arr = np.arange(12).reshape((3, 4))
np.concatenate([arr, arr]) #like rbind in R
np.concatenate([arr, arr], axis=1)  #like cbind in R

With pd, labeled axes enable more generalized concatenation. Extra concerns:
If the objects are indexed differently on the other axes, should we combine their distinct elements or use only the common values?
Do the concatenated chunks of data need to be identifiable as such in the result?
Does the "concatenation axis" have data that needs to be preserved? The default integer labels in a pd.DataFrame are often best discarded when concatenating.<br>
<i>pd.concat()</i> gives a consistent way to address all 3 questions. Given a list of pd.Series with no index overlap, values and indexes are glued together since it works along <i>axis=0</i> by default (like <i>rbind()</i> in R, <i>axis=1</i> is like <i>cbind()</i>). It behaves like an outer join unless <i>join='inner'</i>

In [ ]:
s1 = pd.Series([0, 1], index=["a", "b"], dtype="Int64")
s2 = pd.Series([2, 3, 4], index=["c", "d", "e"], dtype="Int64")
s3 = pd.Series([5, 6], index=["f", "g"], dtype="Int64")
pd.concat(objs=[s1, s2, s3])
pd.concat([s1, s2, s3], axis=1)
s4 = pd.concat([s1, s3])
pd.concat([s1, s4], axis=1, join="inner")

To identify the concatenated pieces in the result, use <i>keys</i>, which adds a level to the index along <i>axis</i>

In [ ]:
result = pd.concat([s1, s1, s3], keys=["one", "one2", "three"])
result
#result.unstack()
pd.concat([s1, s2, s3], axis=1, keys=["one", "two", "three"])

In [ ]:
df1 = pd.DataFrame(np.arange(6).reshape(3, 2), index=["a", "b", "c"],
                   columns=["one", "two"])
df2 = pd.DataFrame(5 + np.arange(4).reshape(2, 2), index=["a", "c"],
                   columns=["three", "four"])
pd.concat([df1, df2], axis=1, keys=["level1", "level2"])

If <i>objs</i> is a dict, use its keys as <i>keys</i>

In [ ]:
pd.concat({"level1": df1, "level2": df2}, axis=1)
pd.concat([df1, df2], axis="columns", keys=["level1", "level2"],
  names=["upper", "lower"]) #name the created axis levels

If the <i>axis</i> index has no relevant data, pass <i>ignore_index=True</i> to discard them before concatenation

In [ ]:
df1 = pd.DataFrame(np.random.standard_normal((3, 4)),
                   columns=["a", "b", "c", "d"])
df2 = pd.DataFrame(np.random.standard_normal((2, 3)),
                   columns=["b", "d", "a"])
pd.concat([df1, df2])
pd.concat([df1, df2], ignore_index=True)

<i>pd.concat()</i> arguments
<table border="1" cellpadding="6" cellspacing="0">
<tr><th>Argument</th><th>Description</th></tr>
<tr><td>objs</td><td>List or dictionary of pandas objects to be concatenated;<br>this is the only required argument</td></tr>
<tr><td>axis</td><td>Axis to concatenate along; defaults to concatenating<br>along rows (axis="index")</td></tr>
<tr><td>join</td><td>Either "inner" or "outer" ("outer" by default); whether<br>to intersect (inner) or union (outer) indexes along the<br>other axes</td></tr>
<tr><td>keys</td><td>Values to associate with objects being concatenated,<br>forming a hierarchical index along the concatenation<br>axis; can be a list or array of arbitrary values, an<br>array of tuples, or a list of arrays (if multiple-level<br>arrays passed in levels)</td></tr>
<tr><td>levels</td><td>Specific indexes to use as hierarchical index level or<br>levels if keys passed</td></tr>
<tr><td>names</td><td>Names for created hierarchical levels if keys and/or<br>levels passed</td></tr>
<tr><td>verify_integrity</td><td>Check new axis in concatenated object for duplicates and<br>raise an exception if so; by default (False) allows duplicates</td></tr>
<tr><td>ignore_index</td><td>Do not preserve indexes along concatenation axis,<br>instead produce a new range(total_length) index</td></tr>
</table>

1 more type of data combination is neither a merge nor concatenation. Given 2 datasets with overlapping indexes, 1 way to replace NAs in 1 with corresponding values in another is via <i>np.where()</i>, but it ignores index alignment (the objects need not even be the same length), which motivates <i>combine_first()</i>. It does the same per column for pd.DataFrames. Think of it as 'patching' NAs. The output has the union of all column names

In [ ]:
a = pd.Series([np.nan, 2.5, 0.0, 3.5, 4.5, np.nan],
              index=["f", "e", "d", "c", "b", "a"])
b = pd.Series([0., np.nan, 2., np.nan, np.nan, 5.],
              index=["a", "b", "c", "d", "e", "f"])
np.where(pd.isna(a), b, a)
a.combine_first(other=b)

In [ ]:
df1 = pd.DataFrame({"a": [1., np.nan, 5., np.nan],
                    "b": [np.nan, 2., np.nan, 6.],
                    "c": range(2, 18, 4)})
df2 = pd.DataFrame({"a": [5., 4., np.nan, 3., 7.],
                    "b": [np.nan, 3., 4., 6., 8.]})
df1.combine_first(df2)

1 common way to rearrange data is via reshape or pivot operations. Recall hierarchical indexing is key. The 2 primary actions: <i>stack()</i> 'rotates' or pivots from columns to rows, adding a level to the index. <i>unstack()</i> pivots from rows to columns. By default, the innermost level is (un)stacked (same with stack). You can give a different <i>level</i> number or name.  The level (un)stacked becomes the lowest level in the result

In [ ]:
data = pd.DataFrame(np.arange(6).reshape((2, 3)),
                    index=pd.Index(["Ohio", "Colorado"], name="state"),
                    columns=pd.Index(["one", "two", "three"],
                    name="number"))
result = data.stack()
result.unstack() #recall: inverse of stack
result.unstack(level=0) #same: level="state"

In [ ]:
df = pd.DataFrame({"left": result, "right": result + 5},
                  columns=pd.Index(["left", "right"], name="side"))
df.unstack(level="state")
df.unstack(level="state").stack("side") #same: level=0

<i>unstack()</i> introduces NAs if values in the level do not overlap perfectly. <i>stack()</i> currently has the <i>dropna</i> option <font color='red'>(will be deprecated in the future)</font>

In [ ]:
s1 = pd.Series([0, 1, 2, 3], index=["a", "b", "c", "d"], dtype="Int64")
s2 = pd.Series([4, 5, 6], index=["c", "d", "e"], dtype="Int64")
data2 = pd.concat([s1, s2], keys=["one", "two"])
data2.unstack()
data2.unstack().stack()
data2.unstack().stack(dropna=False)

Data (eg multiple time series) are commonly stored in DBs/ CSV files in long/ stacked format, where each row represents a single observation

In [ ]:
github_start='https://github.com/wesm/pydata-book/raw/refs/heads/3rd-edition/'
data = pd.read_csv(github_start+"examples/macrodata.csv")
data = data.loc[:, ["year", "quarter", "realgdp", "infl", "unemp"]]
data.columns.name = "item"
data.head()

We use the <i>pd.PeriodIndex</i> <a href=https://pandas.pydata.org/docs/reference/api/pandas.PeriodIndex.html>class</a> to turn the year/ quarter columns into a single level index with datetime of each quarter's start. <i>pop()</i> returns a column while deleting it from the pd.DataFrame

In [ ]:
periods = pd.PeriodIndex.from_fields(year=data.pop("year"),
                         quarter=data.pop("quarter"),)
periods.name="date"
data.index = periods.to_timestamp("D",how='start')
data.head()

Data is frequently stored as below in relational SQL DBs since a fixed schema (column names and data types) allows the number of distinct values in the item column to change if data is added. Below, date and item would likely be the primary keys, offering both relational integrity and easier joins

In [ ]:
long_data = (data.stack()
             .reset_index()
             .rename(columns={0: "value"}))
long_data[:10]

Data can be harder to work with in this format. To get a pd.DataFrame with 1 column per distinct item value indexed by timestamps in the date column, use <i>pivot()</i>, analogous to <i>unstack()</i> except it works on columns, not the index. <i>index</i> and <i>columns</i> give the columns to be used as the row and column index resp. Since it makes an index from <i>index</i>, it is often paired with <i>reset_index()</i>

In [ ]:
pivoted = long_data.pivot(columns="item", index="date", values="value")
pivoted2=long_data.set_index(["date","item"])['value'].unstack(level="item") #same
pivoted.reset_index().head()

 <i>pivot()</i> can reshape multiple value columns simultaneously (output has a MultiIndex), ie <i>values</i> can be a sequence or omitted to use all leftover columns

In [ ]:
#long_data.index.name = None
long_data["value2"] = np.random.standard_normal(len(long_data))
pivoted = long_data.pivot(index="date", columns="item") #same: values=["value", "value2"]
pivoted2 = long_data.set_index(["date", "item"]).unstack(level="item") #same
pivoted.head()
#long_data.pivot(columns="item", index="date", values="value") #same: pivoted["value"]

Use <i>pd.melt()</i> (inverse of <i>pivot()</i>) to go from wide to long, <i>id_vars</i> gives the groups if any, <i>value_vars</i> for columns to unpivot (default all columns except <i>id_vars</i>)

In [ ]:
df = pd.DataFrame({"key": ["foo", "bar", "baz"],
                   "A": [1, 2, 3],
                   "B": [4, 5, 6],
                   "C": [7, 8, 9]})
melted = pd.melt(df, id_vars="key") #same df.melt(id_vars="key")
melted

In [ ]:
pd.melt(df, id_vars="key", value_vars=["A", "B"])
pd.melt(df, value_vars=["A", "B", "C"]) #no grouping variable
pd.melt(df, value_vars=["key", "A", "B"])

In [ ]:
reshaped = melted.pivot(index="key", columns="variable",
                        values="value")
reshaped.reset_index()